In [ ]:
"""
POLYNOMIAL CLASSIFICATION SUITE
Control Group Testing: Raw Neural Network vs Invariant Decision Tree

Fixes implemented:
1. Compares Raw NN (Black Box) against Invariant DT (Mathematical Oracle)
2. Uses 3 Seeds (42, 43, 44) with 5-Fold Stratified Cross Validation
3. Prints 95% Confidence Intervals
4. Implements Explicit Quartic Discriminant
5. Incorporates Degree 5 using the Crit8 critical point features
"""

import numpy as np
import scipy.stats as stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

class PolynomialClassificationSuite:
    
    def __init__(self):
        self.seeds = list(range(20))
        
    # Data generation  
    def generate_polynomial_data(self, degree, n_samples, coef_range=(-10, 10), noise=0.0, seed=42):
        rng = np.random.RandomState(seed)
        if degree == 2: return self._generate_quadratic_data(n_samples, coef_range, noise, rng)
        elif degree == 3: return self._generate_cubic_data(n_samples, coef_range, noise, rng)
        elif degree == 4: return self._generate_quartic_data(n_samples, coef_range, noise, rng)
        elif degree == 5: return self._generate_quintic_data(n_samples, coef_range, noise, rng)
        else: raise ValueError(f"Unsupported degree: {degree}")
            
    def _generate_quadratic_data(self, n_samples, coef_range, noise, rng):
        a = rng.uniform(*coef_range, n_samples)
        b = rng.uniform(*coef_range, n_samples)
        c = rng.uniform(*coef_range, n_samples)
        
        discriminant = b**2 - 4*a*c
        labels = (discriminant < 0).astype(int)
        
        if noise > 0:
            a += rng.normal(0, noise, n_samples)
            b += rng.normal(0, noise, n_samples)
            c += rng.normal(0, noise, n_samples)
            
        X_raw = np.column_stack([a, b, c])
        
        # Explicit discriminant
        clean_discriminant = b**2 - 4*a*c
        X_inv = np.column_stack([X_raw, clean_discriminant]) 
        
        return X_raw, X_inv, labels

    def _generate_cubic_data(self, n_samples, coef_range, noise, rng):
        A = rng.uniform(*coef_range, n_samples)
        B = rng.uniform(*coef_range, n_samples)
        C = rng.uniform(*coef_range, n_samples)
        
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i]])
            labels[i] = 1 if np.any(np.abs(roots.imag) > 1e-10) else 0
            
        if noise > 0:
            A += rng.normal(0, noise, n_samples)
            B += rng.normal(0, noise, n_samples)
            C += rng.normal(0, noise, n_samples)
            
        X_raw = np.column_stack([A, B, C])
        
        # Explicit cubic discriminant
        cubic_discriminant = 18*A*B*C - 4*A**3*C + A**2*B**2 - 4*B**3 - 27*C**2
        X_inv = np.column_stack([X_raw, cubic_discriminant])
        
        return X_raw, X_inv, labels

    def _generate_quartic_data(self, n_samples, coef_range, noise, rng):
        A = rng.uniform(*coef_range, n_samples)
        B = rng.uniform(*coef_range, n_samples)
        C = rng.uniform(*coef_range, n_samples)
        D = rng.uniform(*coef_range, n_samples)
        
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            if n_real >= 3: labels[i] = 0
            elif n_real >= 1: labels[i] = 1
            else: labels[i] = 2
            
        if noise > 0:
            A += rng.normal(0, noise, n_samples)
            B += rng.normal(0, noise, n_samples)
            C += rng.normal(0, noise, n_samples)
            D += rng.normal(0, noise, n_samples)
            
        X_raw = np.column_stack([A, B, C, D])
        
        # Quartic invariants
        I = 12 * 1.0 * D - 3 * A * C + B**2
        J = 72 * 1.0 * B * D + 9 * A * B * C - 27 * 1.0 * C**2 - 27 * A**2 * D - 2 * B**3
        Delta_expr = 4 * I**3 - J**2
        P = 8 * 1.0 * B - 3 * A**2
        D_aux = 64 * 1.0**3 * D - 16 * 1.0**2 * B**2 + 16 * 1.0 * A**2 * B - 16 * 1.0**2 * A * C - 3 * A**4
        
        # Stack all 5 invariants alongside the raw coefficients
        X_inv = np.column_stack([X_raw, I, J, Delta_expr, P, D_aux])
        
        return X_raw, X_inv, labels

    def _generate_quintic_data(self, n_samples, coef_range, noise, rng):
        A = rng.uniform(*coef_range, n_samples)
        B = rng.uniform(*coef_range, n_samples)
        C = rng.uniform(*coef_range, n_samples)
        D = rng.uniform(*coef_range, n_samples)
        E = rng.uniform(*coef_range, n_samples)
        
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            if n_real == 5: labels[i] = 0
            elif n_real == 3: labels[i] = 1
            else: labels[i] = 2
            
        if noise > 0:
            A += rng.normal(0, noise, n_samples)
            B += rng.normal(0, noise, n_samples)
            C += rng.normal(0, noise, n_samples)
            D += rng.normal(0, noise, n_samples)
            E += rng.normal(0, noise, n_samples)
            
        X_raw = np.column_stack([A, B, C, D, E])
        
        # Calculate Crit8 and other critical features
        critical = np.zeros((n_samples, 10))
        for i in range(n_samples):
            try:
                cpts = np.roots([5, 4*A[i], 3*B[i], 2*C[i], D[i]])
                real_cpts = cpts[np.abs(cpts.imag) < 1e-10].real
                critical[i, 0] = len(real_cpts)
                if len(real_cpts) > 0:
                    critical[i, 1:5] = [np.min(real_cpts), np.max(real_cpts), 
                                       np.mean(real_cpts), np.std(real_cpts) if len(real_cpts)>1 else 0]
                    poly = np.poly1d([1, A[i], B[i], C[i], D[i], E[i]])
                    vals = [poly(x) for x in real_cpts]
                    critical[i, 5:8] = [np.min(vals), np.max(vals), np.mean(vals)]
                    nz_vals = [v for v in vals if abs(v) > 1e-10]
                    if len(nz_vals) > 1:
                        critical[i, 8] = sum(1 for j in range(len(nz_vals)-1) 
                                           if nz_vals[j] * nz_vals[j+1] < 0)  # CRIT8!
                infl = np.roots([20, 12*A[i], 6*B[i], 2*C[i]])
                critical[i, 9] = len(infl[np.abs(infl.imag) < 1e-10])
            except: pass
            
        X_inv = np.column_stack([X_raw, critical])
        return X_raw, X_inv, labels


    # Validation tests (20 Seeds, 5-Fold CV)
    def _format_ci(self, scores):
        t_stat = stats.t.ppf(0.975, len(self.seeds)-1)
        mean_score = np.mean(scores)
        ci = t_stat * (np.std(scores, ddof=1) / np.sqrt(len(self.seeds)))
        return f"{mean_score:.3f}±{ci:.3f}"

    def test_extrapolation(self, degree, verbose=True):
        if verbose:
            print(f"\n{'='*60}")
            print(f"EXTRAPOLATION TEST - Degree {degree} (3 Seeds, 5-Fold CV)")
            print(f"{'='*60}")
        
        test_ranges = [10, 20, 50, 100]
        res_nn = {r: [] for r in test_ranges}
        res_dt = {r: [] for r in test_ranges}
        
        for seed in self.seeds:
            X_raw_tr_full, X_inv_tr_full, y_tr_full = self.generate_polynomial_data(degree, 5000, (-10, 10), seed=seed)
            
            test_sets = {}
            for r in test_ranges:
                test_sets[r] = self.generate_polynomial_data(degree, 1000, (-r, r), seed=seed)
                
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            fold_nn, fold_dt = {r: [] for r in test_ranges}, {r: [] for r in test_ranges}
            
            for train_idx, _ in skf.split(X_raw_tr_full, y_tr_full):
                X_r_tr, y_tr = X_raw_tr_full[train_idx], y_tr_full[train_idx]
                X_i_tr = X_inv_tr_full[train_idx] if X_inv_tr_full is not None else None
                
                scaler = StandardScaler()
                nn = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=seed)
                nn.fit(scaler.fit_transform(X_r_tr), y_tr)
                
                dt = DecisionTreeClassifier(max_depth=8, random_state=seed)
                if X_i_tr is not None: dt.fit(X_i_tr, y_tr)
                
                for r in test_ranges:
                    X_r_te, X_i_te, y_te = test_sets[r]
                    fold_nn[r].append(balanced_accuracy_score(y_te, nn.predict(scaler.transform(X_r_te))))
                    if X_i_te is not None:
                        fold_dt[r].append(balanced_accuracy_score(y_te, dt.predict(X_i_te)))
                        
            for r in test_ranges:
                res_nn[r].append(np.mean(fold_nn[r]))
                if fold_dt[r]: res_dt[r].append(np.mean(fold_dt[r]))
                
        for r in test_ranges:
            out = f"  Range ±{r:<3d} | Raw NN: {self._format_ci(res_nn[r])}"
            if res_dt[r]: out += f" | Invariant DT: {self._format_ci(res_dt[r])}"
            if verbose: print(out)

    def test_minimal_training(self, degree, verbose=True):
        if verbose:
            print(f"\n{'='*60}")
            print(f"MINIMAL TRAINING TEST - Degree {degree} (3 Seeds, 5-Fold CV)")
            print(f"{'='*60}")
        
        training_sizes = [25, 50, 100, 500, 1000, 5000] 
        res_nn = {s: [] for s in training_sizes}
        res_dt = {s: [] for s in training_sizes}
        
        for seed in self.seeds:
            # Fixed test set for this seed
            X_raw_te, X_inv_te, y_te = self.generate_polynomial_data(degree, 1000, (-10, 10), seed=seed)
            
            for size in training_sizes:
                X_raw_tr_full, X_inv_tr_full, y_tr_full = self.generate_polynomial_data(degree, size, (-10, 10), seed=seed)
                
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                fold_nn, fold_dt = [], []
                
                try:
                    for train_idx, _ in skf.split(X_raw_tr_full, y_tr_full):
                        X_r_tr, y_tr = X_raw_tr_full[train_idx], y_tr_full[train_idx]
                        X_i_tr = X_inv_tr_full[train_idx] if X_inv_tr_full is not None else None
                        
                        scaler = StandardScaler()
                        nn = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=seed)
                        nn.fit(scaler.fit_transform(X_r_tr), y_tr)
                        fold_nn.append(balanced_accuracy_score(y_te, nn.predict(scaler.transform(X_raw_te))))
                        
                        if X_i_tr is not None:
                            dt = DecisionTreeClassifier(max_depth=8, random_state=seed)
                            dt.fit(X_i_tr, y_tr)
                            fold_dt.append(balanced_accuracy_score(y_te, dt.predict(X_inv_te)))
                except ValueError:
                    # Failsafe if minority class is too small for StratifiedKFold
                    fold_nn = [0.33]; fold_dt = [0.33]
                    
                res_nn[size].append(np.mean(fold_nn))
                if fold_dt: res_dt[size].append(np.mean(fold_dt))
                
        for size in training_sizes:
            out = f"  {size:<4d} samples | Raw NN: {self._format_ci(res_nn[size])}"
            if res_dt[size] and np.mean(res_dt[size]) > 0.35: 
                out += f" | Invariant DT: {self._format_ci(res_dt[size])}"
            if verbose: print(out)

    def test_noise_robustness(self, degree, verbose=True):
        if verbose:
            print(f"\n{'='*60}")
            print(f"NOISE ROBUSTNESS TEST - Degree {degree}")
            print(f"{'='*60}")
        
        noise_levels = [0.0, 0.1, 0.5, 1.0, 2.0]
        res_nn = {n: [] for n in noise_levels}
        res_dt = {n: [] for n in noise_levels}
        
        for seed in self.seeds:
            X_raw_tr_full, X_inv_tr_full, y_tr_full = self.generate_polynomial_data(degree, 6000, (-10, 10), seed=seed)
            
            test_sets = {}
            for noise in noise_levels:
                test_sets[noise] = self.generate_polynomial_data(degree, 1000, (-10, 10), noise=noise, seed=seed)
                
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            fold_nn, fold_dt = {n: [] for n in noise_levels}, {n: [] for n in noise_levels}
            
            for train_idx, _ in skf.split(X_raw_tr_full, y_tr_full):
                X_r_tr, y_tr = X_raw_tr_full[train_idx], y_tr_full[train_idx]
                X_i_tr = X_inv_tr_full[train_idx] if X_inv_tr_full is not None else None
                
                scaler = StandardScaler()
                nn = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=seed)
                nn.fit(scaler.fit_transform(X_r_tr), y_tr)
                
                dt = DecisionTreeClassifier(max_depth=8, random_state=seed)
                if X_i_tr is not None: dt.fit(X_i_tr, y_tr)
                
                for n in noise_levels:
                    X_r_te, X_i_te, y_te = test_sets[n]
                    fold_nn[n].append(balanced_accuracy_score(y_te, nn.predict(scaler.transform(X_r_te))))
                    if X_i_te is not None:
                        fold_dt[n].append(balanced_accuracy_score(y_te, dt.predict(X_i_te)))
                        
            for n in noise_levels:
                res_nn[n].append(np.mean(fold_nn[n]))
                if fold_dt[n]: res_dt[n].append(np.mean(fold_dt[n]))
                
        for n in noise_levels:
            out = f"  Noise σ={n:<3.1f} | Raw NN: {self._format_ci(res_nn[n])}"
            if res_dt[n]: out += f" | Invariant DT: {self._format_ci(res_dt[n])}"
            if verbose: print(out)

    def run_validation_suite(self, degrees=[2, 3, 4, 5]):
        print("\n" + "#"*80)
        print("# Control suite: Raw NN vs Invariant DT")
        print("#"*80)
        
        for degree in degrees:
            print(f"\n{'#'*80}")
            print(f"# Degree {degree}")
            print(f"{'#'*80}")
            self.test_extrapolation(degree)
            self.test_minimal_training(degree)
            self.test_noise_robustness(degree)

if __name__ == "__main__":
    suite = PolynomialClassificationSuite()
    suite.run_validation_suite(degrees=[2,3,4,5])

In [ ]:
"""
QUINTIC FEATURE ENGINEERING
Advanced mathematical features for quintic polynomial classification
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')


class QuinticFeatureExtractor:
    """
    Feature extraction for quintic polynomials.
    
    Implements 6 feature groups (63 total features):
    - Base: 5 (A, B, C, D, E)
    - Sturm: 8
    - Descartes: 6
    - Newton: 10
    - Critical: 10 (includes Crit8)
    - Hybrid: 16
    - Decomposition: 8
    """
    
    def __init__(self):
        pass
    
    # Group 1: Strum sequences (8 features)    
    def sturm_sequence(self, coeffs):
        """Compute Sturm sequence via iterated polynomial division."""
        p = np.poly1d(coeffs)
        p_prime = np.polyder(p)
        sturm_seq = [p, p_prime]
        
        max_iterations = 10
        for _ in range(max_iterations):
            if len(sturm_seq[-1].c) == 0:
                break
            
            _, remainder = np.polydiv(sturm_seq[-2], sturm_seq[-1])
            remainder = -remainder
            
            # Clean up near-zero coefficients
            remainder.c = remainder.c[np.abs(remainder.c) > 1e-10]
            
            if len(remainder.c) == 0 or (len(remainder.c) == 1 and abs(remainder.c[0]) < 1e-10):
                break
            
            sturm_seq.append(remainder)
        
        return sturm_seq
    
    def count_sign_changes(self, sturm_seq, x):
        """Count sign changes in Sturm sequence at point x."""
        signs = []
        for poly in sturm_seq:
            val = poly(x)
            if abs(val) > 1e-10:
                signs.append(np.sign(val))
        
        if len(signs) <= 1:
            return 0
        
        changes = sum(1 for i in range(len(signs)-1) if signs[i] * signs[i+1] < 0)
        return changes
    
    def extract_sturm_features(self, coefficients):
        """Extract 8 Sturm-based features."""
        n_samples = len(coefficients)
        sturm_features = np.zeros((n_samples, 8))
        
        for i in range(n_samples):
            try:
                coeffs = np.concatenate([[1], coefficients[i]])
                sturm_seq = self.sturm_sequence(coeffs)
                
                sturm_features[i, 0] = self.count_sign_changes(sturm_seq, -1000)
                sturm_features[i, 1] = self.count_sign_changes(sturm_seq, 1000)
                sturm_features[i, 2] = abs(sturm_features[i, 0] - sturm_features[i, 1])
                
                test_points = [-10, -1, 0, 1, 10]
                for j, x in enumerate(test_points):
                    sturm_features[i, 3+j] = self.count_sign_changes(sturm_seq, x)
            except:
                sturm_features[i, :] = [2, 2, 3, 2, 2, 2, 2, 2]
        
        return sturm_features

    
    # Group 2: Descartes' rule (6 features)
    def extract_descartes_features(self, coefficients):
        """Extract 6 Descartes-based features."""
        n_samples = len(coefficients)
        descartes_features = np.zeros((n_samples, 6))
        
        for i in range(n_samples):
            poly_coeffs = np.concatenate([[1], coefficients[i]])
            
            # Positive roots
            nonzero_coeffs = poly_coeffs[np.abs(poly_coeffs) > 1e-10]
            sign_changes_pos = sum(1 for j in range(len(nonzero_coeffs)-1) 
                                  if nonzero_coeffs[j] * nonzero_coeffs[j+1] < 0)
            
            # Negative roots (substitute x = -x)
            neg_poly_coeffs = poly_coeffs.copy()
            for j in range(len(neg_poly_coeffs)):
                if (len(neg_poly_coeffs) - 1 - j) % 2 == 1:
                    neg_poly_coeffs[j] *= -1
            
            nonzero_neg = neg_poly_coeffs[np.abs(neg_poly_coeffs) > 1e-10]
            sign_changes_neg = sum(1 for j in range(len(nonzero_neg)-1) 
                                  if nonzero_neg[j] * nonzero_neg[j+1] < 0)
            
            descartes_features[i, 0] = sign_changes_pos
            descartes_features[i, 1] = sign_changes_neg
            descartes_features[i, 2] = sign_changes_pos + sign_changes_neg
            descartes_features[i, 3] = 5 - descartes_features[i, 2]
            descartes_features[i, 4] = sign_changes_pos % 2
            descartes_features[i, 5] = sign_changes_neg % 2
        
        return descartes_features
    
    # Group 3: Newton's sums (10 features)    
    def compute_newton_sums(self, A, B, C, D, E):
        """Compute Newton's sums s1-s5 via Newton's identities."""
        s1 = -A
        s2 = A**2 - 2*B
        s3 = -A**3 + 3*A*B - 3*C
        s4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
        s5 = -A**5 + 5*A**3*B - 5*A*B**2 - 5*A**2*C + 5*B*C + 5*A*D - 5*E
        return s1, s2, s3, s4, s5
    
    def extract_newton_features(self, coefficients):
        """Extract 10 Newton-based features."""
        n_samples = len(coefficients)
        newton_features = np.zeros((n_samples, 10))
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            s1, s2, s3, s4, s5 = self.compute_newton_sums(A, B, C, D, E)
            
            newton_features[i, 0:5] = [s1, s2, s3, s4, s5]
            newton_features[i, 5] = s1 / 5
            newton_features[i, 6] = s2/5 - (s1/5)**2
            newton_features[i, 7] = s3 / (abs(s1) + 1)
            newton_features[i, 8] = s4 / (abs(s2) + 1)
            newton_features[i, 9] = s5 / (abs(s3) + 1)
        
        return newton_features
    
    # Group 4: Critical points (10 features, includes Crit8)    
    def extract_critical_point_features(self, coefficients):
        """Extract 10 critical point features including Crit8."""
        n_samples = len(coefficients)
        critical_features = np.zeros((n_samples, 10))
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            deriv_coeffs = [5, 4*A, 3*B, 2*C, D]
            
            try:
                critical_points = np.roots(deriv_coeffs)
                real_critical = critical_points[np.abs(critical_points.imag) < 1e-10].real
                
                critical_features[i, 0] = len(real_critical)
                
                if len(real_critical) > 0:
                    critical_features[i, 1] = np.min(real_critical)
                    critical_features[i, 2] = np.max(real_critical)
                    critical_features[i, 3] = np.mean(real_critical)
                    critical_features[i, 4] = np.std(real_critical) if len(real_critical) > 1 else 0
                    
                    poly = np.poly1d([1, A, B, C, D, E])
                    critical_values = [poly(x) for x in real_critical]
                    
                    critical_features[i, 5] = np.min(critical_values)
                    critical_features[i, 6] = np.max(critical_values)
                    critical_features[i, 7] = np.mean(critical_values)
                    
                    # CRIT8: Sign changes at critical points
                    nonzero_vals = [v for v in critical_values if abs(v) > 1e-10]
                    if len(nonzero_vals) > 1:
                        sign_changes = sum(1 for j in range(len(nonzero_vals)-1) 
                                         if nonzero_vals[j] * nonzero_vals[j+1] < 0)
                        critical_features[i, 8] = sign_changes
                
                # Inflection points
                second_deriv = [20, 12*A, 6*B, 2*C]
                inflection_roots = np.roots(second_deriv)
                real_inflections = inflection_roots[np.abs(inflection_roots.imag) < 1e-10]
                critical_features[i, 9] = len(real_inflections)
            except:
                critical_features[i, :] = 0
        
        return critical_features
    
    # Group 5: Hybrid symbolic (16 features)
    def create_hybrid_features(self, coefficients):
        """Extract 16 hybrid algebraic features."""
        n_samples = len(coefficients)
        hybrid_features = []
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            
            # Tschirnhaus invariants
            I2 = A**2 - 2*B
            I3 = A**3 - 3*A*B + 3*C
            I4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
            I5 = A**5 - 5*A**3*B + 5*A*B**2 + 5*A**2*C - 5*B*C - 5*A*D + 5*E
            
            # Novel combinations
            S1 = A*B*C - D*E
            S2 = A**2*E - B**2*D + C**3
            S3 = (A*D - B*C)**2 + (B*E - C*D)**2
            S4 = A*C*E - B*D**2
            
            # Discriminant-like differences
            D1 = B**2 - A*C
            D2 = C**2 - B*D
            D3 = D**2 - C*E
            
            # Scale-invariant ratios
            eps = 1e-10
            R1 = (A*E) / (B*D + eps)
            R2 = (B*D) / (C**2 + eps)
            R3 = (A*C*E) / (B*D**2 + eps)
            R4 = I2 / (I3 + eps)
            R5 = I3 / (I4 + eps)
            
            features = [I2, I3, I4, I5, S1, S2, S3, S4, D1, D2, D3, R1, R2, R3, R4, R5]
            hybrid_features.append(features)
        
        return np.array(hybrid_features)
    
    # Group 6: Decomposition (8 features)    
    def extract_decomposition_features(self, coefficients):
        """Extract 8 decomposition/factorization features."""
        n_samples = len(coefficients)
        decomp_features = np.zeros((n_samples, 8))
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            coeffs = [A, B, C, D, E]
            
            decomp_features[i, 0] = np.sum(np.abs(coeffs) < 0.1)
            
            abs_coeffs = np.abs(coeffs)
            decomp_features[i, 1] = np.max(abs_coeffs) / (np.min(abs_coeffs) + 1e-10)
            decomp_features[i, 2] = np.var(abs_coeffs)
            decomp_features[i, 3] = abs(E) < 0.1
            decomp_features[i, 4] = abs(A*E - B*D + C**2) / (abs(A*E) + abs(B*D) + abs(C**2) + 1)
            decomp_features[i, 5] = abs(A**2 - B) / (abs(A**2) + abs(B) + 1)
            decomp_features[i, 6] = abs(B**2 - A*C) / (abs(B**2) + abs(A*C) + 1)
            decomp_features[i, 7] = abs(C**2 - B*D) / (abs(C**2) + abs(B*D) + 1)
        
        return decomp_features
    

    # Main extraction
    def extract_all_features(self, coefficients):
        """Extract all 63 features."""
        sturm_features = self.extract_sturm_features(coefficients)
        descartes_features = self.extract_descartes_features(coefficients)
        newton_features = self.extract_newton_features(coefficients)
        critical_features = self.extract_critical_point_features(coefficients)
        hybrid_features = self.create_hybrid_features(coefficients)
        decomp_features = self.extract_decomposition_features(coefficients)
        
        all_features = np.hstack([
            coefficients,
            sturm_features,
            descartes_features,
            newton_features,
            critical_features,
            hybrid_features,
            decomp_features
        ])
        
        return all_features


class QuinticFeatureComparison:
    """
    Compare baseline vs individual methods vs combined features.
    """
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)
        self.feature_extractor = QuinticFeatureExtractor()
    
    def generate_quintic_data(self, n_samples, coef_range=(-10, 10)):
        """Generate quintic data with proper RNG."""
        A = self.rng.uniform(*coef_range, n_samples)
        B = self.rng.uniform(*coef_range, n_samples)
        C = self.rng.uniform(*coef_range, n_samples)
        D = self.rng.uniform(*coef_range, n_samples)
        E = self.rng.uniform(*coef_range, n_samples)
        
        coefficients = np.column_stack([A, B, C, D, E])
        labels = np.zeros(n_samples, dtype=int)
        
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            
            if n_real == 5:
                labels[i] = 0
            elif n_real == 3:
                labels[i] = 1
            else:
                labels[i] = 2
        
        return coefficients, labels
    
    def test_baseline_single_trial(self, n_samples=5000, verbose=True):
        """Single trial baseline."""
        if verbose:
            print("\n" + "="*60)
            print("Baseline: Raw Coefficients Only (Single Trial)")
            print("="*60)
        
        coefficients, y = self.generate_quintic_data(n_samples)
        
        X_train, X_test, y_train, y_test = train_test_split(
            coefficients, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = MLPClassifier(
            hidden_layer_sizes=(100, 50),
            max_iter=500,
            random_state=42
        )
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Neural Network: Acc={acc:.3f}, Balanced={bal_acc:.3f}")
        
        return bal_acc
    
    def test_baseline_multi_trial(self, n_samples=5000, n_trials=20, verbose=True):
        """
        20-trial baseline validation.
        """
        if verbose:
            print("\n" + "="*60)
            print(f"Baseline: Raw Coefficients Only ({n_trials} trials)")
            print("="*60)
        
        accuracies = []
        
        for trial in range(n_trials):
            # Create new RNG for each trial
            trial_rng = np.random.RandomState(trial)
            
            # Generate data
            A = trial_rng.uniform(-10, 10, n_samples)
            B = trial_rng.uniform(-10, 10, n_samples)
            C = trial_rng.uniform(-10, 10, n_samples)
            D = trial_rng.uniform(-10, 10, n_samples)
            E = trial_rng.uniform(-10, 10, n_samples)
            
            coefficients = np.column_stack([A, B, C, D, E])
            labels = np.zeros(n_samples, dtype=int)
            
            for i in range(n_samples):
                roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                
                if n_real == 5:
                    labels[i] = 0
                elif n_real == 3:
                    labels[i] = 1
                else:
                    labels[i] = 2
            
            # Train-test split
            X_train, X_test, y_train, y_test = train_test_split(
                coefficients, labels, test_size=0.2, random_state=42, stratify=labels
            )
            
            # Train model
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            model = MLPClassifier(
                hidden_layer_sizes=(100, 50),
                max_iter=500,
                random_state=42
            )
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            
            bal_acc = balanced_accuracy_score(y_test, y_pred)
            accuracies.append(bal_acc)
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: {bal_acc:.3f}")
        
        mean_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        
        if verbose:
            print(f"\nMean: {mean_acc:.3f} ± {std_acc:.3f}")
            print(f"Range: [{np.min(accuracies):.3f}, {np.max(accuracies):.3f}]")
        
        return mean_acc, std_acc, accuracies
    
    def test_individual_methods(self, n_samples=5000, verbose=True):
        """Test each feature group individually."""
        if verbose:
            print("\n" + "="*60)
            print("Individual method testing")
            print("="*60)
        
        coefficients, y = self.generate_quintic_data(n_samples)
        
        methods = {
            'Sturm': self.feature_extractor.extract_sturm_features(coefficients),
            'Descartes': self.feature_extractor.extract_descartes_features(coefficients),
            'Newton': self.feature_extractor.extract_newton_features(coefficients),
            'Critical': self.feature_extractor.extract_critical_point_features(coefficients),
            'Hybrid': self.feature_extractor.create_hybrid_features(coefficients),
            'Decomposition': self.feature_extractor.extract_decomposition_features(coefficients)
        }
        
        results = {}
        
        for method_name, features in methods.items():
            X = np.hstack([coefficients, features])
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            model = MLPClassifier(
                hidden_layer_sizes=(100, 50),
                max_iter=500,
                random_state=42
            )
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            
            acc = accuracy_score(y_test, y_pred)
            bal_acc = balanced_accuracy_score(y_test, y_pred)
            
            results[method_name] = {
                'accuracy': acc,
                'balanced_accuracy': bal_acc
            }
            
            if verbose:
                print(f"  {method_name:15s}: Acc={acc:.3f}, Balanced={bal_acc:.3f}")
        
        return results
    
    def test_individual_methods_multi_trial(self, n_samples=5000, n_trials=20, verbose=True):
        """
        Test each feature group individually across 20 trials with 5-Fold CV.
        """
        if verbose:
            print("\n" + "="*60)
            print(f"Individual method testing ({n_trials} trials)")
            print("="*60)
        
        method_names = ['Sturm', 'Descartes', 'Newton', 'Critical', 'Hybrid', 'Decomposition']
        method_accuracies = {name: [] for name in method_names}
        
        for trial in range(n_trials):
            trial_rng = np.random.RandomState(trial)
            
            A = trial_rng.uniform(-10, 10, n_samples)
            B = trial_rng.uniform(-10, 10, n_samples)
            C = trial_rng.uniform(-10, 10, n_samples)
            D = trial_rng.uniform(-10, 10, n_samples)
            E = trial_rng.uniform(-10, 10, n_samples)
            
            coefficients = np.column_stack([A, B, C, D, E])
            labels = np.zeros(n_samples, dtype=int)
            
            for i in range(n_samples):
                roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                
                if n_real == 5: labels[i] = 0
                elif n_real == 3: labels[i] = 1
                else: labels[i] = 2
            
            methods = {
                'Sturm': self.feature_extractor.extract_sturm_features(coefficients),
                'Descartes': self.feature_extractor.extract_descartes_features(coefficients),
                'Newton': self.feature_extractor.extract_newton_features(coefficients),
                'Critical': self.feature_extractor.extract_critical_point_features(coefficients),
                'Hybrid': self.feature_extractor.create_hybrid_features(coefficients),
                'Decomposition': self.feature_extractor.extract_decomposition_features(coefficients)
            }
            
            for method_name, features in methods.items():
                X = np.hstack([coefficients, features])
                
                # 5-Fold CV
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                cv_scores = []
                
                for train_idx, test_idx in skf.split(X, labels):
                    X_train, X_test = X[train_idx], X[test_idx]
                    y_train, y_test = labels[train_idx], labels[test_idx]
                    
                    scaler = StandardScaler()
                    X_train_scaled = scaler.fit_transform(X_train)
                    X_test_scaled = scaler.transform(X_test)
                    
                    model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
                    model.fit(X_train_scaled, y_train)
                    y_pred = model.predict(X_test_scaled)
                    
                    cv_scores.append(balanced_accuracy_score(y_test, y_pred))
                
                method_accuracies[method_name].append(np.mean(cv_scores))
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: ", end="")
                for name in method_names:
                    print(f"{name[:4]}={method_accuracies[name][-1]:.3f} ", end="")
                print()
        
        # Calculate 95% CI
        results = {}
        t_stat = stats.t.ppf(0.975, n_trials-1) if n_trials > 1 else 0
        
        if verbose:
            print(f"\nMean ± CI (across {n_trials} trials):")
        
        for method_name in method_names:
            accs = method_accuracies[method_name]
            mean_acc = np.mean(accs)
            ci_95 = t_stat * (np.std(accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
            
            results[method_name] = {
                'mean': mean_acc,
                'std': ci_95, # Keep dict key as 'std' so run_comprehensive_analysis doesn't throw a KeyError
                'all_accuracies': accs
            }
            
            if verbose:
                print(f"  {method_name:15s}: {mean_acc:.3f} ± {ci_95:.3f}")
        
        return results
    
    def test_combined(self, n_samples=5000, verbose=True):
        """Test all 63 features combined."""
        if verbose:
            print("\n" + "="*60)
            print("Combined: All 63 Features")
            print("="*60)
        
        coefficients, y = self.generate_quintic_data(n_samples)
        X = self.feature_extractor.extract_all_features(coefficients)
        
        if verbose:
            print(f"Total features: {X.shape[1]}")
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = MLPClassifier(
            hidden_layer_sizes=(200, 100, 50),
            max_iter=500,
            random_state=42
        )
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Neural Network: Acc={acc:.3f}, Balanced={bal_acc:.3f}")
        
        return {'accuracy': acc, 'balanced_accuracy': bal_acc}
    
    def run_comprehensive_analysis(self, multi_trial_methods=False):
        """Run all tests."""
        print("="*80)
        print(" Quintic Feature Engineering: Full Test")
        print("="*80)
        
        # 1. Single trial baseline
        single_baseline = self.test_baseline_single_trial()
        
        # 2. Multi-trial baseline
        multi_baseline_mean, multi_baseline_std, _ = self.test_baseline_multi_trial()
        
        # 3. Individual methods
        if multi_trial_methods:
            individual_results = self.test_individual_methods_multi_trial()
        else:
            individual_results = self.test_individual_methods()
        
        # 4. Combined features
        combined_results = self.test_combined()
        
        print("\n" + "="*80)
        print(" SUMMARY")
        print("="*80)
        print(f"Baseline (single trial): {single_baseline:.3f}")
        
        print(f"Baseline (20 trials):    {multi_baseline_mean:.3f} ± {multi_baseline_std:.3f} (95% CI)")
        
        if multi_trial_methods:
            print(f"\nIndividual Methods (20 trials) [95% CI]:")
            for method, result in individual_results.items():
                print(f"  {method:15s}: {result['mean']:.3f} ± {result['std']:.3f}")
        
        print(f"\nCombined (63 features):  {combined_results['balanced_accuracy']:.3f}")
        
        return {
            'baseline_single': single_baseline,
            'baseline_multi_mean': multi_baseline_mean,
            'baseline_multi_std': multi_baseline_std,
            'individual': individual_results,
            'combined': combined_results
        }


# Main execution
if __name__ == "__main__":
    comparison = QuinticFeatureComparison(random_state=42)
    
    results = comparison.run_comprehensive_analysis(multi_trial_methods=True)


In [ ]:
"""
QUINTIC RULE EXTRACTION
"""

import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, balanced_accuracy_score
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# Feature extractor
class QuinticFeatureExtractor:
    def sturm_sequence(self, coeffs):
        p = np.poly1d(coeffs)
        p_prime = np.polyder(p)
        sturm_seq = [p, p_prime]
        for _ in range(10):
            if len(sturm_seq[-1].c) == 0: break
            _, remainder = np.polydiv(sturm_seq[-2], sturm_seq[-1])
            remainder = -remainder
            remainder.c = remainder.c[np.abs(remainder.c) > 1e-10]
            if len(remainder.c) == 0: break
            sturm_seq.append(remainder)
        return sturm_seq
    
    def count_sign_changes(self, sturm_seq, x):
        signs = [np.sign(poly(x)) for poly in sturm_seq if abs(poly(x)) > 1e-10]
        return sum(1 for i in range(len(signs)-1) if signs[i] * signs[i+1] < 0)
    
    def extract_all_features(self, coefficients):
        n = len(coefficients)
        all_features = [coefficients]
        
        # Sturm (8)
        sturm = np.zeros((n, 8))
        for i in range(n):
            try:
                seq = self.sturm_sequence(np.concatenate([[1], coefficients[i]]))
                sturm[i, 0] = self.count_sign_changes(seq, -1000)
                sturm[i, 1] = self.count_sign_changes(seq, 1000)
                sturm[i, 2] = abs(sturm[i, 0] - sturm[i, 1])
                for j, x in enumerate([-10, -1, 0, 1, 10]):
                    sturm[i, 3+j] = self.count_sign_changes(seq, x)
            except: sturm[i, :] = [2, 2, 3, 2, 2, 2, 2, 2]
        all_features.append(sturm)
        
        # Descartes (6)
        descartes = np.zeros((n, 6))
        for i in range(n):
            poly = np.concatenate([[1], coefficients[i]])
            nz = poly[np.abs(poly) > 1e-10]
            sc_pos = sum(1 for j in range(len(nz)-1) if nz[j] * nz[j+1] < 0)
            neg = poly.copy()
            for j in range(len(neg)):
                if (len(neg)-1-j) % 2 == 1: neg[j] *= -1
            nz_neg = neg[np.abs(neg) > 1e-10]
            sc_neg = sum(1 for j in range(len(nz_neg)-1) if nz_neg[j] * nz_neg[j+1] < 0)
            descartes[i] = [sc_pos, sc_neg, sc_pos+sc_neg, 5-(sc_pos+sc_neg), sc_pos%2, sc_neg%2]
        all_features.append(descartes)
        
        # Newton (10)
        newton = np.zeros((n, 10))
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            s1 = -A
            s2 = A**2 - 2*B
            s3 = -A**3 + 3*A*B - 3*C
            s4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
            s5 = -A**5 + 5*A**3*B - 5*A*B**2 - 5*A**2*C + 5*B*C + 5*A*D - 5*E
            newton[i] = [s1, s2, s3, s4, s5, s1/5, s2/5-(s1/5)**2, 
                        s3/(abs(s1)+1), s4/(abs(s2)+1), s5/(abs(s3)+1)]
        all_features.append(newton)
        
        # Critical points (10) 
        critical = np.zeros((n, 10))
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            try:
                cpts = np.roots([5, 4*A, 3*B, 2*C, D])
                real_cpts = cpts[np.abs(cpts.imag) < 1e-10].real
                critical[i, 0] = len(real_cpts)
                if len(real_cpts) > 0:
                    critical[i, 1:5] = [np.min(real_cpts), np.max(real_cpts), 
                                       np.mean(real_cpts), np.std(real_cpts) if len(real_cpts)>1 else 0]
                    poly = np.poly1d([1, A, B, C, D, E])
                    vals = [poly(x) for x in real_cpts]
                    critical[i, 5:8] = [np.min(vals), np.max(vals), np.mean(vals)]
                    nz_vals = [v for v in vals if abs(v) > 1e-10]
                    if len(nz_vals) > 1:
                        critical[i, 8] = sum(1 for j in range(len(nz_vals)-1) 
                                           if nz_vals[j] * nz_vals[j+1] < 0)  # CRIT8!
                infl = np.roots([20, 12*A, 6*B, 2*C])
                critical[i, 9] = len(infl[np.abs(infl.imag) < 1e-10])
            except: pass
        all_features.append(critical)
        
        # Hybrid (16)
        hybrid = []
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            I2 = A**2 - 2*B
            I3 = A**3 - 3*A*B + 3*C
            I4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
            I5 = A**5 - 5*A**3*B + 5*A*B**2 + 5*A**2*C - 5*B*C - 5*A*D + 5*E
            eps = 1e-10
            hybrid.append([I2, I3, I4, I5, A*B*C-D*E, A**2*E-B**2*D+C**3,
                          (A*D-B*C)**2+(B*E-C*D)**2, A*C*E-B*D**2,
                          B**2-A*C, C**2-B*D, D**2-C*E,
                          (A*E)/(B*D+eps), (B*D)/(C**2+eps), (A*C*E)/(B*D**2+eps),
                          I2/(I3+eps), I3/(I4+eps)])
        all_features.append(np.array(hybrid))
        
        # Decomposition (8)
        decomp = np.zeros((n, 8))
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            cs = [A, B, C, D, E]
            abs_cs = np.abs(cs)
            decomp[i] = [np.sum(abs_cs < 0.1), np.max(abs_cs)/(np.min(abs_cs)+1e-10),
                        np.var(abs_cs), abs(E)<0.1,
                        abs(A*E-B*D+C**2)/(abs(A*E)+abs(B*D)+abs(C**2)+1),
                        abs(A**2-B)/(abs(A**2)+abs(B)+1),
                        abs(B**2-A*C)/(abs(B**2)+abs(A*C)+1),
                        abs(C**2-B*D)/(abs(C**2)+abs(B*D)+1)]
        all_features.append(decomp)
        
        return np.hstack(all_features)

# Rule extractor
class QuinticRuleExtractor:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)
        self.feature_extractor = QuinticFeatureExtractor()
        self.model = None
        self.scaler = None
    
    def generate_quintic_data(self, n_samples):
        A = self.rng.uniform(-10, 10, n_samples)
        B = self.rng.uniform(-10, 10, n_samples)
        C = self.rng.uniform(-10, 10, n_samples)
        D = self.rng.uniform(-10, 10, n_samples)
        E = self.rng.uniform(-10, 10, n_samples)
        coefficients = np.column_stack([A, B, C, D, E])
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            labels[i] = 0 if n_real == 5 else (1 if n_real == 3 else 2)
        return self.feature_extractor.extract_all_features(coefficients), labels
    
    def test_crit8_alone(self, n_samples=5000, verbose=True):
        """Test Crit8 performance with just base coefficients."""
        if verbose:
            print("\n" + "="*80)
            print("Crit8 standalone test.")
            print("="*80)
        
        X_all, y = self.generate_quintic_data(n_samples)
        X_base = X_all[:, :5]
        crit8 = X_all[:, 37].reshape(-1, 1)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_base, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)
        
        nn_baseline = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
        nn_baseline.fit(X_train_sc, y_train)
        baseline_acc = balanced_accuracy_score(y_test, nn_baseline.predict(X_test_sc))
        
        if verbose:
            print(f"\nBaseline (5 coefficients only):")
            print(f"  Balanced Accuracy: {baseline_acc:.3f}")
        
        X_with_crit8 = np.hstack([X_base, crit8])
        X_train, X_test, y_train, y_test = train_test_split(
            X_with_crit8, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)
        
        nn_crit8 = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
        nn_crit8.fit(X_train_sc, y_train)
        crit8_acc = balanced_accuracy_score(y_test, nn_crit8.predict(X_test_sc))
        improvement = crit8_acc - baseline_acc
        
        if verbose:
            print(f"\nWith Crit8 (5 coefficients + Crit8):")
            print(f"  Balanced Accuracy: {crit8_acc:.3f}")
            print(f"  Improvement: +{improvement:.3f} ({improvement/baseline_acc*100:.1f}%)")
            print(f"\n*** Adding Crit8 alone improves from {baseline_acc:.1%} to {crit8_acc:.1%} ***")
        
        return {'baseline': baseline_acc, 'with_crit8': crit8_acc, 'improvement': improvement}
    
    def test_crit8_alone_multi_trial(self, n_samples=5000, n_trials=20, verbose=True):
        """Test Crit8 across 20 trials with 5-Fold CV and 95% CIs."""
        if verbose:
            print("\n" + "="*80)
            print(f"Crit8 Standalone test ({n_trials} trials, 5-Fold CV)")
            print("="*80)
        
        baseline_accs, crit8_accs = [], []
        
        for trial in range(n_trials):
            trial_extractor = QuinticRuleExtractor(random_state=trial)
            X_all, y = trial_extractor.generate_quintic_data(n_samples)
            X_base = X_all[:, :5]
            crit8 = X_all[:, 37].reshape(-1, 1)
            
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            cv_base, cv_crit8 = [], []
            
            for train_idx, test_idx in skf.split(X_base, y):
                # Fold Baseline
                X_tr_b, X_te_b = X_base[train_idx], X_base[test_idx]
                y_tr, y_te = y[train_idx], y[test_idx]
                scaler_b = StandardScaler()
                X_tr_b_sc = scaler_b.fit_transform(X_tr_b)
                X_te_b_sc = scaler_b.transform(X_te_b)
                nn_base = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
                nn_base.fit(X_tr_b_sc, y_tr)
                cv_base.append(balanced_accuracy_score(y_te, nn_base.predict(X_te_b_sc)))
                
                # Fold With Crit8
                X_with_crit8 = np.hstack([X_base, crit8])
                X_tr_c, X_te_c = X_with_crit8[train_idx], X_with_crit8[test_idx]
                scaler_c = StandardScaler()
                X_tr_c_sc = scaler_c.fit_transform(X_tr_c)
                X_te_c_sc = scaler_c.transform(X_te_c)
                nn_crit8 = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
                nn_crit8.fit(X_tr_c_sc, y_tr)
                cv_crit8.append(balanced_accuracy_score(y_te, nn_crit8.predict(X_te_c_sc)))
                
            baseline_accs.append(np.mean(cv_base))
            crit8_accs.append(np.mean(cv_crit8))
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: Baseline={baseline_accs[-1]:.3f}, +Crit8={crit8_accs[-1]:.3f}, Δ={crit8_accs[-1]-baseline_accs[-1]:+.3f}")
        
        t_stat = stats.t.ppf(0.975, n_trials-1) if n_trials > 1 else 0
        baseline_mean = np.mean(baseline_accs)
        baseline_ci = t_stat * (np.std(baseline_accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        crit8_mean = np.mean(crit8_accs)
        crit8_ci = t_stat * (np.std(crit8_accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        improvement_mean = crit8_mean - baseline_mean
        
        if verbose:
            print(f"\nNeural Network Results (95% CI):")
            print(f"  Baseline: {baseline_mean:.3f} ± {baseline_ci:.3f}")
            print(f"  +Crit8:   {crit8_mean:.3f} ± {crit8_ci:.3f}")
            print(f"  Improvement: +{improvement_mean:.3f}")
        
        return {
            'baseline_mean': baseline_mean, 'baseline_std': baseline_ci, 
            'crit8_mean': crit8_mean, 'crit8_std': crit8_ci,
            'improvement': improvement_mean,
            'all_baselines': baseline_accs, 'all_crit8': crit8_accs
        }
    
    def test_crit8_decision_tree(self, n_samples=5000, n_trials=20, verbose=True):
        """Test Crit8 with decision trees using 5-Fold CV and 95% CIs."""
        if verbose:
            print("\n" + "="*80)
            print(f"Crit8 with decision trees ({n_trials} trials, 5-Fold CV)")
            print("="*80)
        
        dt_baseline_accs, dt_crit8_accs = [], []
        
        for trial in range(n_trials):
            trial_extractor = QuinticRuleExtractor(random_state=trial)
            X_all, y = trial_extractor.generate_quintic_data(n_samples)
            X_base = X_all[:, :5]
            crit8 = X_all[:, 37].reshape(-1, 1)
            
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            cv_dt_base, cv_dt_crit8 = [], []
            
            for train_idx, test_idx in skf.split(X_base, y):
                # Decision tree baseline
                X_tr_b, X_te_b = X_base[train_idx], X_base[test_idx]
                y_tr, y_te = y[train_idx], y[test_idx]
                dt_baseline = DecisionTreeClassifier(max_depth=8, random_state=42)
                dt_baseline.fit(X_tr_b, y_tr)
                cv_dt_base.append(balanced_accuracy_score(y_te, dt_baseline.predict(X_te_b)))
                
                # Decision tree with crit8
                X_with_crit8 = np.hstack([X_base, crit8])
                X_tr_c, X_te_c = X_with_crit8[train_idx], X_with_crit8[test_idx]
                dt_crit8 = DecisionTreeClassifier(max_depth=8, random_state=42)
                dt_crit8.fit(X_tr_c, y_tr)
                cv_dt_crit8.append(balanced_accuracy_score(y_te, dt_crit8.predict(X_te_c)))
                
            dt_baseline_accs.append(np.mean(cv_dt_base))
            dt_crit8_accs.append(np.mean(cv_dt_crit8))
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: Baseline={dt_baseline_accs[-1]:.3f}, +Crit8={dt_crit8_accs[-1]:.3f}, Δ={dt_crit8_accs[-1]-dt_baseline_accs[-1]:+.3f}")
        
        t_stat = stats.t.ppf(0.975, n_trials-1) if n_trials > 1 else 0
        dt_baseline_mean = np.mean(dt_baseline_accs)
        dt_baseline_ci = t_stat * (np.std(dt_baseline_accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        dt_crit8_mean = np.mean(dt_crit8_accs)
        dt_crit8_ci = t_stat * (np.std(dt_crit8_accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        dt_improvement_mean = dt_crit8_mean - dt_baseline_mean
        
        if verbose:
            print(f"\nDecision Tree Results (95% CI):")
            print(f"  Baseline: {dt_baseline_mean:.3f} ± {dt_baseline_ci:.3f}")
            print(f"  +Crit8:   {dt_crit8_mean:.3f} ± {dt_crit8_ci:.3f}")
            print(f"  Improvement: +{dt_improvement_mean:.3f}")
        
        return {
            'dt_baseline_mean': dt_baseline_mean, 'dt_baseline_std': dt_baseline_ci,
            'dt_crit8_mean': dt_crit8_mean, 'dt_crit8_std': dt_crit8_ci,
            'dt_improvement': dt_improvement_mean,
            'all_dt_baselines': dt_baseline_accs, 'all_dt_crit8': dt_crit8_accs
        }
    
    def train_and_distill(self, n_samples=5000, verbose=True):
        """Single detailed trial with full analysis."""
        if verbose:
            print("\n" + "="*80)
            print("Single-trial run")
            print("="*80)
        
        X, y = self.generate_quintic_data(n_samples)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        if verbose:
            print(f"\nData: {len(X_train)} train, {len(X_test)} test")
            print(f"Features: {X.shape[1]} (5 base + 58 advanced)")
            print("\nTraining Neural Network")
        
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        self.model = MLPClassifier(hidden_layer_sizes=(200, 100, 50), max_iter=500, random_state=42)
        self.model.fit(X_train_scaled, y_train)
        
        y_pred = self.model.predict(X_test_scaled)
        nn_acc = accuracy_score(y_test, y_pred)
        nn_bal = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Neural Network Test Performance:")
            print(f"  Accuracy: {nn_acc:.3f}, Balanced Accuracy: {nn_bal:.3f}")
            print("\nDistilling to Decision Tree")
        
        nn_train_pred = self.model.predict(X_train_scaled)
        nn_test_pred = self.model.predict(X_test_scaled)
        tree = DecisionTreeClassifier(max_depth=8, random_state=42)
        tree.fit(X_train, nn_train_pred)
        tree_test_pred = tree.predict(X_test)
        
        test_fidelity = accuracy_score(nn_test_pred, tree_test_pred)
        tree_test_bal = balanced_accuracy_score(y_test, tree_test_pred)
        
        if verbose:
            print(f"\nDecision Tree Results:")
            print(f"  Fidelity (Test): {test_fidelity:.3f}")
            print(f"  Test Balanced: {tree_test_bal:.3f}")
            print(f"  Tree Depth: {tree.get_depth()}, Leaves: {tree.get_n_leaves()}")
        
        feature_importance = tree.feature_importances_
        top_features = np.argsort(feature_importance)[-10:][::-1]
        feature_names = (['A', 'B', 'C', 'D', 'E'] + 
                        [f'Sturm{i}' for i in range(8)] + [f'Desc{i}' for i in range(6)] +
                        [f'Newton{i}' for i in range(10)] + [f'Crit{i}' for i in range(10)] +
                        [f'Hybrid{i}' for i in range(16)] + [f'Decomp{i}' for i in range(8)])
        
        if verbose:
            print("\nTop 10 Features by Importance:")
            for idx in top_features:
                if feature_importance[idx] > 0.001:
                    print(f"  {feature_names[idx]:12s}: {feature_importance[idx]:.3f}")
            crit8_importance = feature_importance[37]
            print(f"\n*** Crit8 (Feature 37) importance: {crit8_importance:.3f} ***")
            if len(top_features) > 1:
                ratio = feature_importance[top_features[0]] / feature_importance[top_features[1]]
                print(f"Top feature is {ratio:.1f}× more important than 2nd")
        
        return {
            'nn_balanced': nn_bal, 'tree_fidelity': test_fidelity,
            'tree_balanced': tree_test_bal, 'crit8_importance': feature_importance[37]
        }
    
    def run_statistical_validation(self, n_trials=20, n_samples=5000):
        """20-trial statistical validation using 5-Fold Stratified CV and 95% CIs."""
        print("\n" + "="*80)
        print(f"Statistical validation: {n_trials} trials (5-Fold CV)")
        print("="*80)
        
        nn_accs, tree_fids, tree_accs = [], [], []
        
        for trial in range(n_trials):
            extractor = QuinticRuleExtractor(random_state=trial)
            X, y = extractor.generate_quintic_data(n_samples)
            
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            cv_nn_acc, cv_tree_fid, cv_tree_acc = [], [], []
            
            for train_idx, test_idx in skf.split(X, y):
                X_tr, X_te = X[train_idx], X[test_idx]
                y_tr, y_te = y[train_idx], y[test_idx]
                
                scaler = StandardScaler()
                X_tr_sc = scaler.fit_transform(X_tr)
                X_te_sc = scaler.transform(X_te)
                
                # 1. Train Neural Network
                nn = MLPClassifier(hidden_layer_sizes=(200,100,50), max_iter=500, random_state=42)
                nn.fit(X_tr_sc, y_tr)
                nn_test_pred = nn.predict(X_te_sc)
                cv_nn_acc.append(balanced_accuracy_score(y_te, nn_test_pred))
                
                # 2. Distill to Decision Tree (Tree trains on NN's predictions for the TRAINING data)
                nn_train_pred = nn.predict(X_tr_sc)
                tree = DecisionTreeClassifier(max_depth=8, random_state=42)
                tree.fit(X_tr, nn_train_pred) 
                
                # 3. Evaluate Tree (on the held-out test fold)
                tree_test_pred = tree.predict(X_te)
                cv_tree_fid.append(accuracy_score(nn_test_pred, tree_test_pred))
                cv_tree_acc.append(balanced_accuracy_score(y_te, tree_test_pred))
                
            # Average the 5 folds for this specific trial
            nn_accs.append(np.mean(cv_nn_acc))
            tree_fids.append(np.mean(cv_tree_fid))
            tree_accs.append(np.mean(cv_tree_acc))
            
            if trial < 3 or trial == n_trials - 1:
                print(f"Trial {trial+1:2d}: NN={nn_accs[-1]:.3f} Fid={tree_fids[-1]:.3f} Tree={tree_accs[-1]:.3f}")

        # Calculate 95% Confidence Intervals
        t_stat = stats.t.ppf(0.975, n_trials-1) if n_trials > 1 else 0
        
        nn_mean = np.mean(nn_accs)
        nn_ci = t_stat * (np.std(nn_accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        
        fid_mean = np.mean(tree_fids)
        fid_ci = t_stat * (np.std(tree_fids, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        
        tree_mean = np.mean(tree_accs)
        tree_ci = t_stat * (np.std(tree_accs, ddof=1) / np.sqrt(n_trials)) if n_trials > 1 else 0.0
        
        print("\n" + "="*80)
        print("Statistical summary (Mean ± 95% CI)")
        print("="*80)
        print(f"Neural Network Test Balanced Acc: {nn_mean:.3f} ± {nn_ci:.3f}")
        print(f"Decision Tree Test Fidelity:      {fid_mean:.3f} ± {fid_ci:.3f}")
        print(f"Decision Tree Test Balanced Acc:  {tree_mean:.3f} ± {tree_ci:.3f}")
        print("="*80)
        
        return {
            'nn': (nn_mean, nn_ci, nn_accs),
            'fidelity': (fid_mean, fid_ci, tree_fids),
            'tree': (tree_mean, tree_ci, tree_accs)
        }
    
    def run_complete_analysis(self, test_crit8=True, multi_trial_crit8=True,
                             test_crit8_dt=True, detailed=True, multi_trial=True, 
                             n_trials=20, n_samples=5000):
        """Run complete analysis with all tests."""
        print("\n" + "#"*80)
        print("# Quintic rule extraction")
        print("#"*80)
        
        results = {}
        if test_crit8:
            results['crit8_single'] = self.test_crit8_alone(n_samples=n_samples, verbose=True)
        if multi_trial_crit8:
            results['crit8_multi'] = self.test_crit8_alone_multi_trial(
                n_samples=n_samples, n_trials=n_trials, verbose=True
            )
        if test_crit8_dt:
            results['crit8_dt'] = self.test_crit8_decision_tree(
                n_samples=n_samples, n_trials=n_trials, verbose=True
            )
        if detailed:
            results['detailed'] = self.train_and_distill(n_samples=n_samples, verbose=True)
        if multi_trial:
            results['statistical'] = self.run_statistical_validation(
                n_trials=n_trials, n_samples=n_samples
            )
        return results

# Run it
if __name__ == "__main__":
    print("Starting Quintic Rule Extraction Analysis")
    
    extractor = QuinticRuleExtractor(random_state=42)
    results = extractor.run_complete_analysis(
        test_crit8=True,
        multi_trial_crit8=True,
        test_crit8_dt=True,
        detailed=True,
        multi_trial=True,
        n_trials=20,
        n_samples=5000
    )
    
    print("\nResults Summary:")
    print("-" * 40)
    
    if 'crit8_single' in results:
        print(f"Crit8 (seed=42): {results['crit8_single']['baseline']:.1%} → {results['crit8_single']['with_crit8']:.1%}")
    
    if 'crit8_multi' in results:
        print(f"Crit8 (20 trials): {results['crit8_multi']['baseline_mean']:.1%}±{results['crit8_multi']['baseline_std']:.1%} → {results['crit8_multi']['crit8_mean']:.1%}±{results['crit8_multi']['crit8_std']:.1%}")
        print(f"  Average improvement: +{results['crit8_multi']['improvement']:.1%}")
    
    if 'crit8_dt' in results:
        print(f"\nCrit8 Decision Trees (20 trials): {results['crit8_dt']['dt_baseline_mean']:.1%}±{results['crit8_dt']['dt_baseline_std']:.1%} → {results['crit8_dt']['dt_crit8_mean']:.1%}±{results['crit8_dt']['dt_crit8_std']:.1%}")
        print(f"  Average improvement: +{results['crit8_dt']['dt_improvement']:.1%}")
    
    if 'statistical' in results:
        nn_mean, nn_std, _ = results['statistical']['nn']
        fid_mean, fid_std, _ = results['statistical']['fidelity']
        tree_mean, tree_std, _ = results['statistical']['tree']
        print(f"\nNN: {nn_mean:.1%}±{nn_std:.1%}  |  Fidelity: {fid_mean:.1%}±{fid_std:.1%}  |  Tree: {tree_mean:.1%}±{tree_std:.1%}")